In [12]:
import pandas as pd
from pathlib import Path
import numpy as np
from scipy.signal import find_peaks
import os
os.listdir("../data/preprocessed/biosignals")

['Person10_D1_2_ID_2.csv',
 'Person11_D1_2_ID_3.csv',
 'Person12_D1_2_ID_4.csv',
 'Person13_D1_2_ID_5.csv',
 'Person14_D1_2_ID_6.csv',
 'Person15_D1_3_ID_1.csv',
 'Person16_D1_3_ID_2.csv',
 'Person17_D1_3_ID_3.csv',
 'Person18_D1_3_ID_4.csv',
 'Person19_D1_4_ID_1.csv',
 'Person1_D1_1_ID_1.csv',
 'Person20_D1_4_ID_2.csv',
 'Person21_D1_4_ID_3.csv',
 'Person22_D1_4_ID_4.csv',
 'Person23_D1_5_ID_1.csv',
 'Person24_D1_5_ID_2.csv',
 'Person25_D1_6_ID_1.csv',
 'Person26_D1_6_ID_2.csv',
 'Person2_D1_1_ID_2.csv',
 'Person3_D1_1_ID_3.csv',
 'Person4_D1_1_ID_4.csv',
 'Person5_D1_1_ID_5.csv',
 'Person6_D1_1_ID_6.csv',
 'Person7_D1_1_ID_7.csv',
 'Person8_D1_1_ID_8.csv',
 'Person9_D1_2_ID_1.csv']

## Compute the mean and std of the EDA for 30s chunks

In [ ]:
# Setup paths based on your group's directory structure
REPO_ROOT = Path(r"C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis")
# REPO_ROOT = Path(__file__).resolve().parent.parent
BIOSIGNAL_DIR = REPO_ROOT / "data" / "preprocessed" / "biosignals"
FEATURE_DIR = REPO_ROOT / "data" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = '30s'

# helper function
def temp_slope(series):
    """Calculates the linear slope of a 30-second window."""
    series = series.dropna()
    if len(series) < 2:
        return np.nan
    x = np.arange(len(series))
    # np.polyfit returns [slope, intercept], we just want slope.
    return np.polyfit(x, series.values, 1)[0]

# helper function
def eda_peaks(series):
    """Counts the number of peaks in the EDA signal for a 30-second window."""
    series = series.dropna()
    peaks, _ = find_peaks(series.values)
    return len(peaks)

def extract_features(file_path: Path) -> pd.DataFrame:
    """Reads a person's preprocessed biosignal file and extracts 30s window features."""
    
    # 1. Load data and set the datetime index
    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')
    
    # Extract the person ID from the filename (e.g., "Person1_D1_1_1234")
    person_id = file_path.stem
    
    # 2. Group by round and phase to prevent "bleeding" across experimental boundaries
    # Then resample the time index into 30-second tumbling windows
    windowed = df.groupby(['round', 'phase']).resample(WINDOW_SIZE)
    
    # 3. Calculate statistics for each window
    # We start with just EDA mean and standard deviation
    features = windowed.agg({
        'HR': ['mean', 'std', 'max', 'min'], # Add a helper function to get the frequency domain HR signal
        'EDA': ['mean', 'std', 'max', 'min', eda_peaks],
        'TEMP': [temp_slope]
    })
    
    # 4. Clean up the multi-level columns created by .agg()
    # This turns ('EDA', 'mean') into 'eda_mean'
    features.columns = [f"{col[0].lower()}_{col[1]}" for col in features.columns]
    
    # 5. Clean up the index
    # Resampling creates windows where there might be no data
    features = features.dropna()
    features = features.reset_index()
    
    # Add our subject identifier back in
    features.insert(0, 'subject_id', person_id)
    
    return features

def build_feature_dataset():
    print(f"Scanning for preprocessed biosignals in {BIOSIGNAL_DIR}...")
    
    all_features = []
    processed_count = 0
    
    for file_path in BIOSIGNAL_DIR.glob("*.csv"):
        person_features = extract_features(file_path)
        all_features.append(person_features)
        processed_count += 1
        print(f"Processed features for: {file_path.stem} ({len(person_features)} windows)")
        
    if not all_features:
        print("No CSV files found. Check your BIOSIGNAL_DIR path.")
        return
        
    # Combine all subjects into one master dataset
    final_dataset = pd.concat(all_features, ignore_index=True)
    
    # Save
    output_path = FEATURE_DIR / "biosignal_features_30s.csv"
    final_dataset.to_csv(output_path, index=False)
    
    print("\n--- Feature Extraction Complete ---")
    print(f"Total subjects processed: {processed_count}")
    print(f"Total 30-second windows generated: {len(final_dataset)}")
    print(f"Dataset saved to: {output_path}")

build_feature_dataset()

Scanning for preprocessed biosignals in C:\Users\Bruger\Documents\DTU\Semester 3\02582 Computational Data Analysis\Case 2 - Git\biosignal-analysis\data\preprocessed\biosignals...
Processed features for: Person10_D1_2_ID_2 (134 windows)
Processed features for: Person11_D1_2_ID_3 (132 windows)
Processed features for: Person12_D1_2_ID_4 (132 windows)
Processed features for: Person13_D1_2_ID_5 (134 windows)
Processed features for: Person14_D1_2_ID_6 (131 windows)
Processed features for: Person15_D1_3_ID_1 (130 windows)
Processed features for: Person16_D1_3_ID_2 (130 windows)
Processed features for: Person17_D1_3_ID_3 (129 windows)
Processed features for: Person18_D1_3_ID_4 (129 windows)
Processed features for: Person19_D1_4_ID_1 (134 windows)
Processed features for: Person1_D1_1_ID_1 (145 windows)
Processed features for: Person20_D1_4_ID_2 (134 windows)
Processed features for: Person21_D1_4_ID_3 (133 windows)
Processed features for: Person22_D1_4_ID_4 (134 windows)
Processed features for: 